# Layer Redundancy & CKA: Which Layers Are Redundant?

## Analyzing Llama 3.1 8B Layer Similarity with Centered Kernel Alignment

**Paper References:**
- Kornblith et al., *"Similarity of Neural Network Representations Revisited"* (ICML 2019)
- Men et al., *"ShortGPT: Layers in LLMs are More Redundant Than You Expect"* (2024)

---

### What is CKA (Centered Kernel Alignment)?

**Centered Kernel Alignment (CKA)** is a similarity metric for comparing representations learned by different layers (or different networks). Unlike naive metrics such as correlation or cosine similarity on raw activations, CKA is:

- **Invariant to orthogonal transformations** — it captures structural similarity, not just element-wise alignment
- **Invariant to isotropic scaling** — rescaling all activations uniformly does not change the score
- **Effective across different dimensionalities** — can compare layers with different hidden sizes

**Linear CKA** uses a linear kernel and computes similarity via the Hilbert-Schmidt Independence Criterion (HSIC):

$$\text{CKA}(X, Y) = \frac{\| Y^\top X \|_F^2}{\| X^\top X \|_F \cdot \| Y^\top Y \|_F}$$

Where $X \in \mathbb{R}^{n \times p}$ and $Y \in \mathbb{R}^{n \times q}$ are activation matrices (samples x features).

**Key insight from ShortGPT:** Many layers in large language models produce nearly identical representations to their inputs, meaning they could be pruned with minimal loss in performance. We will verify this by:
1. Computing the full 32x32 CKA similarity matrix across all layers
2. Measuring Block Importance (BI) via cosine similarity between layer input/output
3. Measuring KL divergence impact when skipping individual layers

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import os
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM

%matplotlib inline

matplotlib.rcParams["figure.figsize"] = (12, 8)
matplotlib.rcParams["font.size"] = 12

MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"
CACHE_DIR = os.path.expanduser("~/models")
OUTPUT_DIR = os.path.expanduser("~/hook_outputs/charts")
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## Load Model and Tokenizer

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, cache_dir=CACHE_DIR)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    cache_dir=CACHE_DIR,
    torch_dtype=torch.float16,
    device_map="auto",
)
model.eval()

num_layers = model.config.num_hidden_layers
print("Number of layers: {}".format(num_layers))
print("Hidden size: {}".format(model.config.hidden_size))
print("Model loaded successfully.")

## Step 1: Capture Hidden States from All 32 Layers

We register forward hooks on every decoder layer to capture the hidden state outputs.
Each hidden state is flattened from `(batch, seq_len, hidden_dim)` to `(batch*seq_len, hidden_dim)` for CKA computation.

In [ ]:
prompt = "The key to understanding deep neural networks is to analyze how information flows through each layer of the model."

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
print("Prompt: '{}'".format(prompt))
print("Token count: {}".format(inputs["input_ids"].shape[1]))

# Storage for hidden states: layer index -> flattened activations
hidden_states = {}
hook_handles = []

def make_hook(layer_idx):
    def hook_fn(module, input, output):
        # output is a tuple; first element is the hidden state
        h = output[0].detach().float()
        # Handle both 2D (seq, hidden) and 3D (batch, seq, hidden) shapes
        if h.dim() == 2:
            hidden_states[layer_idx] = h.cpu()
        elif h.dim() == 3:
            batch, seq, hidden = h.shape
            hidden_states[layer_idx] = h.reshape(batch * seq, hidden).cpu()
    return hook_fn

# Register hooks on all decoder layers
for i, layer in enumerate(model.model.layers):
    handle = layer.register_forward_hook(make_hook(i))
    hook_handles.append(handle)

# Also capture the embedding output (layer 0 input)
embed_state = {}

def embed_hook(module, input, output):
    h = output.detach().float()
    if h.dim() == 2:
        embed_state["embed"] = h.cpu()
    elif h.dim() == 3:
        batch, seq, hidden = h.shape
        embed_state["embed"] = h.reshape(batch * seq, hidden).cpu()

embed_handle = model.model.embed_tokens.register_forward_hook(embed_hook)
hook_handles.append(embed_handle)

# Forward pass
with torch.no_grad():
    outputs = model(**inputs)

# Remove hooks
for handle in hook_handles:
    handle.remove()

print("Captured hidden states from {} layers + embedding".format(len(hidden_states)))
for idx in sorted(hidden_states.keys())[:3]:
    print("  Layer {}: shape {}".format(idx, hidden_states[idx].shape))

## Step 2: Implement Linear CKA

Linear CKA measures representational similarity using the Hilbert-Schmidt Independence Criterion (HSIC) with a linear kernel. It is computed as:

$$\text{CKA}(X, Y) = \frac{\text{HSIC}(X, Y)}{\sqrt{\text{HSIC}(X, X) \cdot \text{HSIC}(Y, Y)}}$$

In [ ]:
def linear_CKA(X, Y):
    """
    Compute Linear CKA between two activation matrices.
    X: (n, p) - n samples, p features
    Y: (n, q) - n samples, q features
    Returns: scalar CKA similarity in [0, 1]
    """
    XTX = X.T @ X
    YTY = Y.T @ Y
    YTX = Y.T @ X
    hsic_xy = (YTX ** 2).sum()
    hsic_xx = (XTX ** 2).sum()
    hsic_yy = (YTY ** 2).sum()
    return (hsic_xy / (hsic_xx.sqrt() * hsic_yy.sqrt())).item()

# Quick sanity check: CKA of a matrix with itself should be 1.0
test_matrix = torch.randn(10, 64)
print("CKA(X, X) = {:.6f} (should be 1.0)".format(linear_CKA(test_matrix, test_matrix)))

## Step 3: Compute the 32x32 CKA Similarity Matrix

We compute pairwise CKA between all layer outputs. The resulting matrix reveals which layers learn similar representations. Block-diagonal structure indicates groups of layers that are functionally similar.

In [ ]:
cka_matrix = np.zeros((num_layers, num_layers))

print("Computing {}x{} CKA matrix...".format(num_layers, num_layers))
for i in range(num_layers):
    for j in range(i, num_layers):
        cka_val = linear_CKA(hidden_states[i], hidden_states[j])
        cka_matrix[i, j] = cka_val
        cka_matrix[j, i] = cka_val
    if (i + 1) % 8 == 0:
        print("  Completed row {}/{}".format(i + 1, num_layers))

print("CKA matrix computed.")
print("Diagonal values (should be ~1.0): {}".format(
    np.diag(cka_matrix)[:5].round(4)
))

## Visualization 1: CKA Similarity Heatmap

The heatmap shows representational similarity between all pairs of layers. Block-diagonal structure indicates groups of layers that compute similar transformations — these are candidates for pruning.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

im = ax.imshow(cka_matrix, cmap="magma", vmin=0, vmax=1, origin="lower")
ax.set_xlabel("Layer Index")
ax.set_ylabel("Layer Index")
ax.set_title("Linear CKA Similarity Between Llama 3.1 8B Layers")
ax.set_xticks(range(0, num_layers, 4))
ax.set_yticks(range(0, num_layers, 4))

cbar = plt.colorbar(im, ax=ax, shrink=0.8)
cbar.set_label("CKA Similarity")

plt.tight_layout()
save_path = os.path.join(OUTPUT_DIR, "chart_14_cka_heatmap.png")
plt.savefig(save_path, dpi=150, bbox_inches="tight")
print("Saved: {}".format(save_path))
plt.show()

## Step 4: Block Importance (BI) Score

Following ShortGPT (Men et al., 2024), we compute the **Block Importance** score for each layer. This measures how much a layer transforms its input by computing the cosine similarity between the layer's input and output hidden states.

- **High cosine similarity** (close to 1.0) = the layer barely changes the representation = **redundant**
- **Low cosine similarity** = the layer significantly transforms the representation = **important**

We define layer importance as `1 - cosine_similarity(input, output)`.

In [ ]:
# Capture both input and output of each layer for BI score
layer_inputs = {}
layer_outputs = {}
hook_handles = []

def make_bi_hook(layer_idx):
    def hook_fn(module, input, output):
        # input is a tuple; first element is the hidden state
        inp = input[0].detach().float()
        out = output[0].detach().float()
        # Handle both 2D (seq, hidden) and 3D (batch, seq, hidden) shapes
        if inp.dim() == 2:
            layer_inputs[layer_idx] = inp.cpu()
            layer_outputs[layer_idx] = out.cpu()
        elif inp.dim() == 3:
            batch, seq, hidden = inp.shape
            layer_inputs[layer_idx] = inp.reshape(batch * seq, hidden).cpu()
            layer_outputs[layer_idx] = out.reshape(batch * seq, hidden).cpu()
    return hook_fn

for i, layer in enumerate(model.model.layers):
    handle = layer.register_forward_hook(make_bi_hook(i))
    hook_handles.append(handle)

with torch.no_grad():
    _ = model(**inputs)

for handle in hook_handles:
    handle.remove()

# Compute cosine similarity and importance for each layer
cosine_sims = []
importances = []

for i in range(num_layers):
    inp = layer_inputs[i]
    out = layer_outputs[i]
    # Average cosine similarity across all token positions
    cos_sim = F.cosine_similarity(inp, out, dim=-1).mean().item()
    cosine_sims.append(cos_sim)
    importances.append(1.0 - cos_sim)

print("Layer importance scores (1 - cosine_sim):")
for i in range(num_layers):
    marker = " <-- most redundant" if importances[i] == min(importances) else ""
    marker = " <-- most important" if importances[i] == max(importances) else marker
    print("  Layer {:2d}: cosine_sim={:.4f}, importance={:.4f}{}".format(
        i, cosine_sims[i], importances[i], marker
    ))

## Visualization 2: Layer Importance Bar Chart

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

colors = plt.cm.RdYlGn(np.array(importances) / max(importances))
bars = ax.bar(range(num_layers), importances, color=colors, edgecolor="black", linewidth=0.5)

ax.set_xlabel("Layer Index")
ax.set_ylabel("Importance Score (1 - cosine similarity)")
ax.set_title("Layer Importance in Llama 3.1 8B (ShortGPT Block Importance)")
ax.set_xticks(range(0, num_layers, 2))
ax.axhline(y=np.mean(importances), color="red", linestyle="--", alpha=0.7,
           label="Mean importance = {:.4f}".format(np.mean(importances)))
ax.legend()
ax.grid(axis="y", alpha=0.3)

# Annotate most and least important
most_important = np.argmax(importances)
least_important = np.argmin(importances)
ax.annotate("Most important\n(Layer {})".format(most_important),
            xy=(most_important, importances[most_important]),
            xytext=(most_important + 2, importances[most_important] + 0.01),
            arrowprops=dict(arrowstyle="->", color="darkgreen"),
            fontsize=9, color="darkgreen", fontweight="bold")
ax.annotate("Most redundant\n(Layer {})".format(least_important),
            xy=(least_important, importances[least_important]),
            xytext=(least_important + 2, importances[least_important] + 0.02),
            arrowprops=dict(arrowstyle="->", color="darkred"),
            fontsize=9, color="darkred", fontweight="bold")

plt.tight_layout()
save_path = os.path.join(OUTPUT_DIR, "chart_15_layer_importance.png")
plt.savefig(save_path, dpi=150, bbox_inches="tight")
print("Saved: {}".format(save_path))
plt.show()

## Step 5: Skip-Layer Analysis

We test what happens when we skip each layer one at a time. For each layer, we register a hook that replaces the layer's output with its input (effectively making the layer an identity function). We then measure the KL divergence between the original output distribution and the skip-layer output distribution.

**High KL divergence** = skipping that layer causes significant change = **important layer**
**Low KL divergence** = skipping that layer barely matters = **redundant layer**

In [ ]:
# First, get the original output logits as reference
with torch.no_grad():
    original_logits = model(**inputs).logits  # (batch, seq, vocab)
    original_probs = F.softmax(original_logits[:, -1, :].float(), dim=-1)  # last token probs

print("Original output shape: {}".format(original_logits.shape))
print("Reference probabilities captured for last token position.")

# Now skip each layer one at a time and measure KL divergence
kl_divergences = []

def make_skip_hook(layer_idx):
    """Hook that passes input through unchanged, effectively skipping the layer."""
    def hook_fn(module, input, output):
        # Modify hidden states in-place: copy input into output
        inp_hidden = input[0]
        out_hidden = output[0]
        # Handle shape mismatch: input may be 3D [batch, seq, hidden]
        # while output may be 2D [seq, hidden] or vice versa
        if inp_hidden.shape == out_hidden.shape:
            out_hidden.data.copy_(inp_hidden.data)
        elif inp_hidden.dim() == 3 and out_hidden.dim() == 2:
            out_hidden.data.copy_(inp_hidden.squeeze(0).data)
        elif inp_hidden.dim() == 2 and out_hidden.dim() == 3:
            out_hidden.data.copy_(inp_hidden.unsqueeze(0).data)
        else:
            out_hidden.data.copy_(inp_hidden.reshape(out_hidden.shape).data)
        # No return needed — in-place modification propagates automatically
    return hook_fn

print("\nComputing KL divergence for skipping each layer...")
for skip_idx in range(num_layers):
    # Register skip hook on the target layer
    handle = model.model.layers[skip_idx].register_forward_hook(make_skip_hook(skip_idx))

    with torch.no_grad():
        skip_logits = model(**inputs).logits
        skip_probs = F.softmax(skip_logits[:, -1, :].float(), dim=-1)

    handle.remove()

    # KL divergence: D_KL(original || skip)
    kl_div = F.kl_div(
        skip_probs.log(), original_probs, reduction="batchmean"
    ).item()
    kl_divergences.append(kl_div)

    if (skip_idx + 1) % 8 == 0:
        print("  Completed {}/{} layers".format(skip_idx + 1, num_layers))

print("\nKL divergence when skipping each layer:")
for i in range(num_layers):
    marker = ""
    if kl_divergences[i] == max(kl_divergences):
        marker = " <-- most impactful to skip"
    elif kl_divergences[i] == min(kl_divergences):
        marker = " <-- least impactful to skip"
    print("  Layer {:2d}: KL = {:.6f}{}".format(i, kl_divergences[i], marker))

## Visualization 3: KL Divergence When Skipping Each Layer

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

# Color bars by KL divergence magnitude (red = high impact, green = low impact / safe to skip)
max_kl = max(kl_divergences)
norm_kl = [kl / max_kl if max_kl > 0 else 0 for kl in kl_divergences]
colors = plt.cm.RdYlGn_r(norm_kl)

bars = ax.bar(range(num_layers), kl_divergences, color=colors, edgecolor="black", linewidth=0.5)

ax.set_xlabel("Skipped Layer Index")
ax.set_ylabel("KL Divergence from Original Output")
ax.set_title("Impact of Skipping Each Layer in Llama 3.1 8B (KL Divergence)")
ax.set_xticks(range(0, num_layers, 2))
ax.axhline(y=np.mean(kl_divergences), color="blue", linestyle="--", alpha=0.7,
           label="Mean KL = {:.4f}".format(np.mean(kl_divergences)))
ax.legend()
ax.grid(axis="y", alpha=0.3)

# Annotate extremes
most_impact = np.argmax(kl_divergences)
least_impact = np.argmin(kl_divergences)
ax.annotate("Highest impact\n(Layer {})".format(most_impact),
            xy=(most_impact, kl_divergences[most_impact]),
            xytext=(most_impact - 5, kl_divergences[most_impact] * 0.85),
            arrowprops=dict(arrowstyle="->", color="red"),
            fontsize=9, color="red", fontweight="bold")
ax.annotate("Safest to skip\n(Layer {})".format(least_impact),
            xy=(least_impact, kl_divergences[least_impact]),
            xytext=(least_impact + 3, max(kl_divergences) * 0.3),
            arrowprops=dict(arrowstyle="->", color="green"),
            fontsize=9, color="green", fontweight="bold")

plt.tight_layout()
save_path = os.path.join(OUTPUT_DIR, "chart_16_skip_layer.png")
plt.savefig(save_path, dpi=150, bbox_inches="tight")
print("Saved: {}".format(save_path))
plt.show()

## Summary & Key Findings

### Three Complementary Views of Layer Redundancy

| Analysis | Method | What It Reveals |
|----------|--------|-----------------|
| **CKA Heatmap** | Pairwise representational similarity | Block-diagonal structure shows groups of layers computing similar transformations |
| **Block Importance (BI)** | Cosine similarity between layer input/output | Layers with high input-output similarity are redundant (they barely change the representation) |
| **Skip-Layer KL Divergence** | KL divergence when bypassing a layer | Layers with low KL divergence can be safely removed without changing the model's output distribution |

### Expected Patterns in Llama 3.1 8B

1. **Early layers (0-3)** tend to be highly important -- they handle tokenization-to-representation mapping and basic syntactic processing
2. **Middle layers (~10-25)** often show high mutual CKA similarity and lower importance scores, suggesting functional redundancy
3. **Final layers (28-31)** are critical for output generation and typically show high KL divergence when skipped
4. **The CKA heatmap** should reveal block-diagonal structure, indicating that nearby layers learn similar representations

### Implications for Model Compression

These results directly inform layer pruning strategies like ShortGPT:
- Layers with **low BI scores** (high cosine similarity) are prime candidates for removal
- Layers with **low skip-KL divergence** can be pruned with minimal performance degradation
- The **CKA block structure** suggests which groups of layers could be merged or shared

### References
- Kornblith et al., "Similarity of Neural Network Representations Revisited" (ICML 2019)
- Men et al., "ShortGPT: Layers in LLMs are More Redundant Than You Expect" (2024)